# 水墨電台 · Colab 打包 APK

直接在 Google Colab 把專案編譯成安卓 APK。

**步驟**：
1. 依序執行下方程式碼格。
2. 出現上傳提示時，把**整個專案資料夾壓成 zip** 上傳（需含 `ink_radio_kivy.py`、`buildozer.spec`、`extra_manifest.xml`、`java/`；其中 `BootReceiver.java` 位於 `java/org/inkradio/BootReceiver.java`）。
3. 等待 buildozer 自動下載 SDK/NDK 並編譯（首次約 15–40 分鐘，**建議用 Colab Pro**；免費版常因逾時/斷線失敗）。
4. 最後一格會自動下載 `*.apk`。

> **避免斷線技巧**：打包格會即時滾動輸出；請讓瀏覽器分頁保持開啟，並定時回來看進度。若免費版還是斷，建議訂閱 Colab Pro。

> 注意：免費版 Colab 磁碟/記憶體有限，若失敗可改本地 Linux 或 Docker（見 README_packaging.md）。


In [ ]:
# 安裝系統依賴與 buildozer
!apt-get update -qq
!apt-get install -qq -y python3-pip python3-setuptools git zip unzip openjdk-17-jdk autoconf libtool pkg-config zlib1g-dev libncurses5-dev ffmpeg libsdl2-dev libsdl2-image-dev libsdl2-mixer-dev libsdl2-ttf-dev libgstreamer1.0-dev gstreamer1.0-plugins-base gstreamer1.0-plugins-good gstreamer1.0-plugins-bad
!pip install -q buildozer

In [ ]:
# 上傳專案 zip（只上傳 inkradio_colab_build.zip，勿同時選 .ipynb 筆記本）
from google.colab import files
uploaded = files.upload()
zips = [n for n in uploaded if n.lower().endswith('.zip')]
if not zips:
    raise FileNotFoundError('沒偵測到 .zip 檔，請只上傳 inkradio_colab_build.zip。目前上傳的檔名：' + str(list(uploaded.keys())))
zip_name = zips[0]
print('已上傳:', zip_name)


In [ ]:
# 解壓縮專案，並自動定位真正含 buildozer.spec 的目錄
import zipfile, os, shutil
if not zipfile.is_zipfile(zip_name):
    raise RuntimeError('檔案不是有效的 zip：' + zip_name + '。請確認上傳的是 inkradio_colab_build.zip。')
proj = '/content/inkradio'
if os.path.exists(proj):
    shutil.rmtree(proj)
os.makedirs(proj, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(proj)

# 遞迴尋找 buildozer.spec，把專案根目錄內容上移到 proj
def find_spec(root):
    for dirpath, _, filenames in os.walk(root):
        if 'buildozer.spec' in filenames:
            return dirpath
    return None

real = find_spec(proj)
if real and os.path.abspath(real) != os.path.abspath(proj):
    for name in os.listdir(real):
        shutil.move(os.path.join(real, name), os.path.join(proj, name))

print('專案檔:', sorted(os.listdir(proj)))
print('BootReceiver 路徑存在:', os.path.isfile(os.path.join(proj, 'java', 'org', 'inkradio', 'BootReceiver.java')))


In [ ]:
# 檢查專案檔案完整度（沿用上方解壓縮後的 proj 路徑）
import os
os.chdir(proj)

# 必備檔案清單
required = {
    'ink_radio_kivy.py': 'ink_radio_kivy.py',
    'buildozer.spec': 'buildozer.spec',
    'extra_manifest.xml': 'extra_manifest.xml',
    'BootReceiver.java': os.path.join('java', 'org', 'inkradio', 'BootReceiver.java'),
}

# 若 BootReceiver.java 被放到根目錄，自動移到正確的 java/org/inkradio/ 下
root_boot = os.path.join(proj, 'BootReceiver.java')
target_dir = os.path.join(proj, 'java', 'org', 'inkradio')
target_boot = os.path.join(target_dir, 'BootReceiver.java')
if os.path.isfile(root_boot):
    os.makedirs(target_dir, exist_ok=True)
    shutil.move(root_boot, target_boot)
    print('已將 BootReceiver.java 移動到正確路徑:', target_boot)

missing = []
for label, rel in required.items():
    full = os.path.join(proj, rel)
    if os.path.exists(full):
        print(f'檔案已就緒：{label}')
    else:
        print(f'缺失檔案：{label}')
        missing.append(label)

if missing:
    raise FileNotFoundError('專案檔案不完整，請確保 zip 內含上述缺失檔案。')
else:
    print('所有必要檔案齊全，準備打包。')


In [ ]:
# 執行打包（debug APK）。首次會下載 Android SDK/NDK，請耐心等候。
# 輸出會即時滾動；若網路慢請改用 Colab Pro 避免逾時。
import os
os.chdir('/content/inkradio')
!buildozer android debug


In [ ]:
# 下載產出的 APK
import glob, os
apks = glob.glob('/content/inkradio/bin/*.apk')
if apks:
    print('找到 APK:', apks)
    files.download(apks[0])
else:
    print('未找到 APK，請檢查上一步的編譯輸出。')